# Spatial and temporal fleet analysis

This notebook explores the H3 hourly aggregates produced by `src/etl.py`, alongside route and stop data. It identifies dispatch-wait bottlenecks, compares traffic speeds by hour, and maps H3 congestion cells. The generated data is synthetic and is intended to demonstrate the analysis workflow.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
import pydeck as pdk

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in candidates if (path / "src" / "etl.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not locate src/etl.py from the notebook working directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
hourly_path = PROCESSED_DIR / "hex_hourly.parquet"
summary_path = PROCESSED_DIR / "hex_summary.parquet"
if not hourly_path.exists() or not summary_path.exists():
    raise FileNotFoundError("Spatial aggregates are missing; run `python -m src.etl` from the project root first.")

hourly = pd.read_parquet(hourly_path)
summary = pd.read_parquet(summary_path)
routes = pd.read_parquet(RAW_DIR / "routes.parquet")
print(f"{len(summary):,} H3 summary rows | {len(hourly):,} H3-hour rows | {len(routes):,} routes")
display(hourly.head())

## Dispatch-wait bottlenecks

Wait occurs at the depot before a route starts. Calculate wait directly from route-level records, where each route contributes once, rather than counting its repeated GPS pings.

In [ ]:
depot_wait = routes.groupby("depot", observed=True).agg(
    routes=("route_id", "nunique"),
    mean_wait_min=("wait_min", "mean"),
    median_wait_min=("wait_min", "median"),
    p90_wait_min=("wait_min", lambda values: values.quantile(0.90)),
).sort_values("mean_wait_min", ascending=False)
display(depot_wait.round(2))

fig = px.bar(
    depot_wait.reset_index(),
    x="depot",
    y="mean_wait_min",
    error_y=depot_wait.reset_index()["p90_wait_min"] - depot_wait.reset_index()["mean_wait_min"],
    title="Average dispatch wait by depot (error bar reaches p90)",
    labels={"depot": "Depot", "mean_wait_min": "Mean wait (minutes)"},
)
fig.show()

## Congestion by hour

`speed_ratio` compares moving speed with the free-flow speed for the same vehicle type. The fleet-level hourly value is weighted by moving minutes and is therefore not a simple mean of hexagon averages.

In [ ]:
by_hour = (
    hourly.assign(weighted_speed=hourly["moving_min"] * hourly["speed_ratio"])
    .groupby("hour", observed=True)
    .agg(weighted_speed=("weighted_speed", "sum"), moving_min=("moving_min", "sum"))
    .assign(speed_ratio=lambda frame: frame["weighted_speed"] / frame["moving_min"])
    .reset_index()
)
px.line(
    by_hour,
    x="hour",
    y="speed_ratio",
    markers=True,
    title="Fleet speed relative to vehicle-specific free flow by hour",
    labels={"hour": "Hour of day", "speed_ratio": "Speed ratio (1 = free flow)"},
).show()

## Explore congestion on the H3 map

Choose an hour, region, and vehicle type. Lower speed ratios indicate slower traffic. The HTML map is written to `data/processed/fleet_map.html`.

In [ ]:
HOUR = 17
REGION = "Center"
VEHICLE_TYPE = "van"
map_data = hourly.loc[
    (hourly["hour"] == HOUR)
    & (hourly["region"] == REGION)
    & (hourly["vehicle_type"] == VEHICLE_TYPE)
].copy()
if map_data.empty:
    raise ValueError(f"No H3 cells found for hour={HOUR}, region={REGION!r}, vehicle={VEHICLE_TYPE!r}.")

map_data["congestion"] = (1 - map_data["speed_ratio"].clip(0, 1)).clip(0, 1)
layer = pdk.Layer(
    "H3HexagonLayer",
    map_data,
    get_hexagon="h3",
    get_fill_color="[255 * congestion, 200 * (1 - congestion), 60, 190]",
    extruded=False,
    pickable=True,
    auto_highlight=True,
)
view = pdk.ViewState(latitude=52.52, longitude=13.405, zoom=9.5, pitch=35)
deck = pdk.Deck(
    layers=[layer],
    initial_view_state=view,
    tooltip={"text": "Region: {region}\nVehicle: {vehicle_type}\nHour: {hour}\nSpeed ratio: {speed_ratio}\nRoutes: {n_routes}"},
)
map_path = PROCESSED_DIR / "fleet_map.html"
deck.to_html(str(map_path), open_browser=False)
print(f"Saved map to {map_path}")
display(map_data[["h3", "region", "vehicle_type", "hour", "n_routes", "speed_ratio"]].sort_values("speed_ratio").head(10))

## Takeaways

Use the depot table to identify dispatch queues, the hourly chart to locate broad peak periods, and the H3 map to find localized slow zones. Notebook 02 analyzes delay attribution, route-level outliers, and statistical differences by algorithm and dispatch window. Interpret all findings as patterns in simulated data, not proof of causal effects in a live fleet.